I am importing Pandas for data processing and datetime for creating timestamps. These libraries are required for data transformation and logging.

In [0]:
from datetime import datetime
import pandas as pd

def log_pipeline_result(stage, status, message):

    timestamp = datetime.now()
    timestamp_text = timestamp.strftime("%Y%m%d_%H%M%S")

    log_path = (
        "/Volumes/customersprocess/default/"
        f"customer_sales_logs/"
        f"{stage}_{status}_{timestamp_text}.csv"
    )

    log_data = pd.DataFrame([
        {
            "timestamp": timestamp,
            "pipeline": "customer_sales_etl",
            "stage": stage,
            "status": status,
            "message": message
        }
    ])

    log_data.to_csv(log_path, index=False)

    print(f"Log written: {log_path}")

In [0]:
try:

    # Your existing Silver code
    # Read Bronze
    # Clean data
    # Run validations
    # Referential integrity checks
    # Write Silver

    log_pipeline_result(
        "silver",
        "SUCCESS",
        "Silver transformation completed successfully"
    )

except Exception as e:

    log_pipeline_result(
        "silver",
        "FAILED",
        str(e)
    )

    raise

Log written: /Volumes/customersprocess/default/customer_sales_logs/silver_SUCCESS_20260914_144131.csv


im reading the bronze files in silver notbook so we can do our transformations in this notebook

In [0]:
import pandas as pd

bronze_base = "/Volumes/customersprocess/default/customer_sales_bronze"

customer_path = f"{bronze_base}/customer.csv"
orders_path = f"{bronze_base}/orders.csv"
sales_path = f"{bronze_base}/sales.csv"

customer_df = pd.read_csv(customer_path)
orders_df = pd.read_csv(orders_path)
sales_df = pd.read_csv(sales_path)

print("Bronze files loaded successfully")

Bronze files loaded successfully


I am converting column names to lowercase and removing unnecessary spaces. This keeps the schema consistent.

In [0]:
customer_df.columns = customer_df.columns.str.lower().str.strip()
orders_df.columns = orders_df.columns.str.lower().str.strip()
sales_df.columns = sales_df.columns.str.lower().str.strip()

print("Column names standardized")

Column names standardized


In [0]:
print(customer_df.columns.tolist())# its used for convert a pandas series index or dataframe column into python native list
print(orders_df.columns.tolist())
print(sales_df.columns.tolist())

['customer_id', 'customer_name', 'email', 'city', 'segment']
['order_id', 'customer_id', 'order_date', 'product_id', 'quantity', 'order_status']
['order_id', 'product_id', 'unit_price', 'discount', 'sales_amount', 'payment_method']


Here I am standardizing the column names by removing unnecessary spaces and converting them to lowercase. The purpose is to make the column names consistent. This becomes useful when different source files have different capitalization or unwanted spaces in their column names.

In [0]:
customer_df["customer_name"] = (
    customer_df["customer_name"]
    .astype(str)
    .str.strip()
)

customer_df["email"] = (
    customer_df["email"]
    .astype(str)
    .str.strip()
    .str.lower()
)

customer_df["city"] = (
    customer_df["city"]
    .astype(str)
    .str.strip()
)

customer_df["segment"] = (
    customer_df["segment"]
    .astype(str)
    .str.strip()
)

orders_df["order_status"] = (
    orders_df["order_status"]
    .astype(str)
    .str.strip()
)

sales_df["payment_method"] = (
    sales_df["payment_method"]
    .astype(str)
    .str.strip()
)

print("Text fields standardized")

Text fields standardized


Here I am converting the order_date column into a proper date format. I also convert fields such as quantity, unit_price, discount, and sales_amount into numeric values. I use errors="coerce" so that if an invalid value is found, Pandas converts it into a missing value instead of immediately stopping the process. Our data quality checks can then identify that invalid value.

In [0]:
orders_df["order_date"] = pd.to_datetime(
    orders_df["order_date"],
    errors="coerce"
)

orders_df["quantity"] = pd.to_numeric(
    orders_df["quantity"],
    errors="coerce"
)

sales_df["unit_price"] = pd.to_numeric(
    sales_df["unit_price"],
    errors="coerce"
)

sales_df["discount"] = pd.to_numeric(
    sales_df["discount"],
    errors="coerce"
)

sales_df["sales_amount"] = pd.to_numeric(
    sales_df["sales_amount"],
    errors="coerce"
)

print("Data types standardized")

Data types standardized


Here I am checking whether the relationships between the three datasets are correct. Every customer_id in the orders data should exist in the customer data. Every order_id in the sales data should exist in the orders data. I also check that the product_id is consistent between the orders and sales data for the same order. This is important because a join can technically work even when the underlying relationships are incorrect.

In [0]:
customer_ids = set(customer_df["customer_id"])

invalid_customer_ids = orders_df[
    ~orders_df["customer_id"].isin(customer_ids)
]

if len(invalid_customer_ids) > 0:

    print(
        "FAIL: Orders contains customer IDs "
        "that do not exist in Customer"
    )

    print(
        "Invalid rows:",
        len(invalid_customer_ids)
    )

else:

    print(
        "PASS: Every order customer_id "
        "exists in Customer"
    )

PASS: Every order customer_id exists in Customer


In [0]:
orders_df["customer_id"].isin(customer_ids)

0      True
1      True
2      True
3      True
4      True
       ... 
995    True
996    True
997    True
998    True
999    True
Name: customer_id, Length: 1000, dtype: bool

In [0]:
order_ids = set(orders_df["order_id"])

invalid_order_ids = sales_df[
    ~sales_df["order_id"].isin(order_ids)
]

if len(invalid_order_ids) > 0:

    print(
        "FAIL: Sales contains order IDs "
        "that do not exist in Orders"
    )

    print(
        "Invalid rows:",
        len(invalid_order_ids)
    )

else:

    print(
        "PASS: Every sales order_id "
        "exists in Orders"
    )

PASS: Every sales order_id exists in Orders


In [0]:
sales_order_check = sales_df.merge(
    orders_df[["order_id", "product_id"]],
    on="order_id",
    how="left",
    suffixes=("_sales", "_orders")
)

invalid_product_mapping = sales_order_check[
    sales_order_check["product_id_sales"]
    != sales_order_check["product_id_orders"]
]

if len(invalid_product_mapping) > 0:

    print(
        "FAIL: Sales product_id does not match "
        "Orders product_id"
    )

    print(
        "Invalid rows:",
        len(invalid_product_mapping)
    )

else:

    print(
        "PASS: Sales product_id matches "
        "Orders product_id"
    )

PASS: Sales product_id matches Orders product_id


After cleaning and validating the data, I write the cleaned DataFrames into the Silver location. At this point, the Silver layer contains standardized and cleaner data that is ready for the Gold transformation.

In [0]:
silver_base = "/Volumes/customersprocess/default/customer_sales_silver"

silver_customer_path = f"{silver_base}/customer.csv"
silver_orders_path = f"{silver_base}/orders.csv"
silver_sales_path = f"{silver_base}/sales.csv"

print("Silver paths ready")

Silver paths ready


In [0]:
customer_df.to_csv(
    silver_customer_path,
    index=False
)

orders_df.to_csv(
    silver_orders_path,
    index=False
)

sales_df.to_csv(
    silver_sales_path,
    index=False
)

print("Silver files written successfully")

Silver files written successfully
